In [0]:
CREATE TABLE IF NOT EXISTS cfpb_risk.app.issues (
  issue_id STRING,
  alert_id STRING,
  institution_display_name STRING,
  rssd_id STRING,
  complaint_month DATE,
  product STRING,
  issue STRING,
  risk_score BIGINT,
  alert_level STRING,
  alert_reason STRING,
  current_status STRING,
  priority STRING,
  issue_owner STRING,
  due_date DATE,
  latest_note STRING,
  created_ts TIMESTAMP,
  updated_ts TIMESTAMP
)
USING DELTA;

CREATE TABLE IF NOT EXISTS cfpb_risk.app.issue_events (
  event_id STRING,
  issue_id STRING,
  alert_id STRING,
  institution_display_name STRING,
  event_ts TIMESTAMP,
  event_type STRING,
  event_user STRING,
  old_status STRING,
  new_status STRING,
  old_priority STRING,
  new_priority STRING,
  old_owner STRING,
  new_owner STRING,
  old_due_date DATE,
  new_due_date DATE,
  note_text STRING,
  event_comment STRING
)
USING DELTA;

MERGE INTO cfpb_risk.app.issues t
USING (
  SELECT
    sha2(concat_ws('|', alert_id, 'issue'), 256) AS issue_id,
    alert_id,
    institution_display_name,
    rssd_id,
    complaint_month,
    product,
    issue,
    risk_score,
    alert_level,
    alert_reason,
    'New' AS current_status,
    CASE
      WHEN alert_level = 'Urgent' THEN 'High'
      WHEN alert_level = 'High' THEN 'Medium'
      ELSE 'Low'
    END AS priority,
    current_timestamp() AS load_ts
  FROM cfpb_risk.gold.risk_alerts
  WHERE alert_level IN ('High', 'Urgent')
) s
ON t.alert_id = s.alert_id
WHEN MATCHED THEN UPDATE SET
  t.institution_display_name = s.institution_display_name,
  t.rssd_id = s.rssd_id,
  t.complaint_month = s.complaint_month,
  t.product = s.product,
  t.issue = s.issue,
  t.risk_score = s.risk_score,
  t.alert_level = s.alert_level,
  t.alert_reason = s.alert_reason,
  t.updated_ts = s.load_ts
WHEN NOT MATCHED THEN INSERT(
  issue_id,
  alert_id,
  institution_display_name,
  rssd_id,
  complaint_month,
  product,
  issue,
  risk_score,
  alert_level,
  alert_reason,
  current_status,
  priority,
  issue_owner,
  due_date,
  latest_note,
  created_ts,
  updated_ts
)
VALUES (
  s.issue_id,
  s.alert_id,
  s.institution_display_name,
  s.rssd_id,
  s.complaint_month,
  s.product,
  s.issue,
  s.risk_score,
  s.alert_level,
  s.alert_reason,
  s.current_status,
  s.priority,
  NULL,
  NULL,
  NULL,
  s.load_ts,
  s.load_ts
);

INSERT INTO cfpb_risk.app.issue_events(
    event_id,
    issue_id,
    alert_id,
    institution_display_name,
    event_ts,
    event_type,
    event_user,
    old_status,
    new_status,
    old_priority,
    new_priority,
    old_owner,
    new_owner,
    old_due_date,
    new_due_date,
    note_text,
    event_comment
)
SELECT
  sha2(concat_ws('|', i.issue_id, 'CREATED'), 256) AS event_id,
  i.issue_id,
  i.alert_id,
  i.institution_display_name,
  i.created_ts AS event_ts,
  'CREATED' AS event_type,
  'system' AS event_user,
  NULL AS old_status,
  i.current_status AS new_status,
  NULL AS old_priority,
  i.priority AS new_priority,
  NULL AS old_owner,
  i.issue_owner AS new_owner,
  NULL AS old_due_date,
  i.due_date AS new_due_date,
  i.latest_note AS note_text,
  'Issue created from risk alert seeding process' AS event_comment
FROM cfpb_risk.app.issues i
WHERE NOT EXISTS(
  SELECT 1
  FROM cfpb_risk.app.issue_events e 
  WHERE e.issue_id = i.issue_id
  AND e.event_type = 'CREATED'
);